1. Загрузите данные из файла data-logistic.csv

In [43]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

data = pd.read_csv('data-logistic.csv', header=None)
X = data.iloc[:, 1:].values
y = data.iloc[:, 0].values

3. Реализуйте градиентный спуск для обычной и L2-регуляризованной
(с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения
используйте вектор (0, 0).

In [44]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, C=0, k=0.1, max_iter=10000, tol=1e-5, w1=0.0, w2=0.0):
    n = len(y)

    for iteration in range(max_iter):
        # Линейные комбинации: y_i * (w1*x1_i + w2*x2_i)
        margins = y * (w1 * X[:, 0] + w2 * X[:, 1])

        # Общий множитель: y_i * (1 - sigmoid(margin_i))
        common = y * (1 - sigmoid(margins))

        # Градиентный шаг
        new_w1 = w1 + k * (1/n) * np.mean(common * X[:, 0]) - k * C * w1
        new_w2 = w2 + k * (1/n) * np.mean(common * X[:, 1]) - k * C * w2

        # Проверка сходимости (евклидово расстояние между итерациями)
        dist = np.sqrt((new_w1 - w1)**2 + (new_w2 - w2)**2)
        w1, w2 = new_w1, new_w2

        if dist < tol:
            print(f"Сходимость достигнута на итерации {iteration + 1}")
            break
    else:
        print(f"Достигнут лимит итераций ({max_iter})")

    return w1, w2

def predict_proba(X, w1, w2):
    return sigmoid(w1 * X[:, 0] + w2 * X[:, 1])

4. Запустите градиентный спуск и доведите до сходимости (евклидово
расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). 

In [45]:
w1_noreg, w2_noreg = gradient_descent(X, y, C=0, k=0.1)
proba_noreg = predict_proba(X, w1_noreg, w2_noreg)



w1_reg, w2_reg = gradient_descent(X, y, C=10, k=0.1)
proba_reg = predict_proba(X, w1_reg, w2_reg)


Сходимость достигнута на итерации 5665
Сходимость достигнута на итерации 2


5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? 

In [46]:
auc_noreg = roc_auc_score(y, proba_noreg)
auc_reg = roc_auc_score(y, proba_reg)
print(f"{auc_noreg} {auc_reg}")

0.936095238095238 0.9362857142857142


6.  Попробуйте поменять длину шага. Будет ли сходиться алгоритм,
если делать более длинные шаги? Как меняется число итераций
при уменьшении длины шага?


In [47]:
for k in [0.01, 0.1, 0.5, 1.0]:
    w1, w2 = gradient_descent(X, y, C=0, k=k)
    proba = predict_proba(X, w1, w2)
    auc = roc_auc_score(y, proba)
    print(f"k={k}: AUC-ROC={auc:.3f}")
print("Не будет, число итераций уменьшается")

Сходимость достигнута на итерации 7635
k=0.01: AUC-ROC=0.937
Сходимость достигнута на итерации 5665
k=0.1: AUC-ROC=0.936
Сходимость достигнута на итерации 3629
k=0.5: AUC-ROC=0.929
Сходимость достигнута на итерации 2412
k=1.0: AUC-ROC=0.928
Не будет, число итераций уменьшается


7. Попробуйте менять начальное приближение. Влияет ли оно на чтонибудь?


In [48]:
def gradient_descent_with_init(X, y, w1_init=0.0, w2_init=0.0, C=0, k=0.1, max_iter=10000):
    w1, w2 = w1_init, w2_init
    x1 = X[:, 0]
    x2 = X[:, 1]
    for _ in range(max_iter):
        pred = y * (w1 * x1 + w2 * x2)
        error = 1 - sigmoid(pred)
        w1_new = w1 + k * np.mean(y * x1 * error) - k * C * w1
        w2_new = w2 + k * np.mean(y * x2 * error) - k * C * w2
        if np.sqrt((w1_new - w1) ** 2 + (w2_new - w2) ** 2) < 1e-5:
            break
        w1, w2 = w1_new, w2_new
    return w1, w2

for w1_init, w2_init in [(0, 0), (1, 1), (-1, -1), (0.5, -0.5)]:
    w1_i, w2_i = gradient_descent_with_init(X, y, w1_init=w1_init, w2_init=w2_init)
    auc_i = roc_auc_score(y, sigmoid(w1_i * X[:, 0] + w2_i * X[:, 1]))
    print(f"init=({w1_init}, {w2_init}): AUC={auc_i:.3f}, w=({w1_i:.4f}, {w2_i:.4f})")

init=(0, 0): AUC=0.927, w=(0.2878, 0.0920)
init=(1, 1): AUC=0.927, w=(0.2878, 0.0920)
init=(-1, -1): AUC=0.927, w=(0.2878, 0.0920)
init=(0.5, -0.5): AUC=0.927, w=(0.2884, 0.0914)
